# Video Dubbing Pipeline (VoxCPM): English → Chinese

Dub English videos into Chinese using **[VoxCPM](https://github.com/OpenBMB/VoxCPM)** for TTS.

This reuses every non-TTS step from `dub_pipeline.py` (vocal separation, ASR + diarization,
subtitle-driven / LLM translation, duration alignment, ffmpeg assembly) and only swaps the
speech-synthesis engine to VoxCPM via `voxcpm_dub_pipeline.py`.

**Features:**
- Voice cloning from the original speaker (timbre + prosody) — no separate emotion model
- Speaker diarization (2-4 speakers)
- Subtitle-driven mode: if a complete `*.zh-cn.srt` sits next to the video, its text is used verbatim (no LLM)
- Background audio preservation, duration-aligned output, checkpoint/resume

**Prerequisites:**
1. Set `HF_TOKEN` and `LLM_API_KEY` in [Colab Secrets](https://colab.research.google.com/notebooks/secrets.ipynb)
2. Accept pyannote terms: https://huggingface.co/pyannote/speaker-diarization-3.1 and https://huggingface.co/pyannote/segmentation-3.0
3. GPU runtime with ~8GB+ VRAM (VoxCPM 2B). T4/L4/A100 all work.

## 0. Mount Google Drive & Set Model Cache

All large model files (VoxCPM, Whisper, Demucs, Pyannote) are cached on Google Drive so they persist across sessions.

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

# --- Model cache on Google Drive (avoids re-downloading every session) ---
DRIVE_CACHE = "/content/drive/MyDrive/index-tts-cache"
os.makedirs(f"{DRIVE_CACHE}/hf_home", exist_ok=True)
os.makedirs(f"{DRIVE_CACHE}/torch_home", exist_ok=True)

# HuggingFace models: VoxCPM, Whisper, Pyannote, WhisperX alignment, CAMPPlus
os.environ["HF_HOME"] = f"{DRIVE_CACHE}/hf_home"
# PyTorch hub models: Demucs
os.environ["TORCH_HOME"] = f"{DRIVE_CACHE}/torch_home"

print(f"Model cache: {DRIVE_CACHE}")
print(f"HF_HOME:    {os.environ['HF_HOME']}")
print(f"TORCH_HOME: {os.environ['TORCH_HOME']}")

## 1. Install dependencies

In [ ]:
!nvidia-smi
import sys
print(f"Python {sys.version}")

In [ ]:
# Clone the repo (contains both dub_pipeline.py and voxcpm_dub_pipeline.py)
%cd /content
!git clone -b py3.12 https://github.com/deluxebear/index-tts.git 2>/dev/null || (cd /content/index-tts && git pull)
%cd /content/index-tts

In [ ]:
%cd /content/index-tts

# Uninstall conflicting Colab packages + install project (for the indextts package:
# CAMPPlus speaker embedding is imported from indextts by the reused pipeline code)
!pip uninstall -y torch torchvision torchaudio tensorflow keras tensorboard protobuf 2>/dev/null
!pip install ninja
!pip install -e ".[webui]" torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu128

# Dubbing pipeline deps
!apt-get update -qq && apt-get install -y -qq rubberband-cli > /dev/null
!pip install -q demucs whisperx pyrubberband soundfile openai

# VoxCPM TTS engine
!pip install -q voxcpm


# Restart runtime so new packages take effect (numpy especially)
print("Restarting runtime to apply package changes...")
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

### After runtime restart, run this cell to restore environment

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

DRIVE_CACHE = "/content/drive/MyDrive/index-tts-cache"
os.environ["HF_HOME"] = f"{DRIVE_CACHE}/hf_home"
os.environ["TORCH_HOME"] = f"{DRIVE_CACHE}/torch_home"

%cd /content/index-tts

import torch, numpy, numba
print(f"torch={torch.__version__} cuda={torch.cuda.is_available()}")
print(f"numpy={numpy.__version__}")
print(f"numba={numba.__version__}")
print("All good!")

## 2. Settings

VoxCPM weights auto-download from HuggingFace on first model load (cached on Drive via `HF_HOME`). No manual checkpoint download needed — unlike the IndexTTS2 pipeline.

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')        # HuggingFace (free) — for speaker diarization
LLM_API_KEY = userdata.get('LLM_API_KEY')  # LLM API key — for translation (skipped if subtitles provided)

# --- LLM provider (uncomment one) ---
LLM_API_BASE = "https://api.openai.com/v1"       ; LLM_MODEL = "gpt-4o-mini"
# LLM_API_BASE = "https://api.deepseek.com/v1"    ; LLM_MODEL = "deepseek-chat"
# LLM_API_BASE = "https://generativelanguage.googleapis.com/v1beta/openai/" ; LLM_MODEL = "gemini-2.0-flash"

# --- VoxCPM settings ---
MODEL_ID = "openbmb/VoxCPM2"   # or "openbmb/VoxCPM-0.5B" (smaller/faster)
CFG_VALUE = 2.0                # guidance scale
INFERENCE_TIMESTEPS = 10       # denoising steps
ULTIMATE_CLONE = False         # prompt_wav_path+prompt_text cloning on long segments

# --- Dubbing settings ---
NUM_SPEAKERS = 2               # Expected number of speakers (2-4)
WORK_DIR = "dub_workspace"     # Intermediate files (can be a Google Drive path)
CLEANUP = False                # Delete intermediate files after completion

print(f"VoxCPM: {MODEL_ID} (cfg={CFG_VALUE}, steps={INFERENCE_TIMESTEPS}, ultimate_clone={ULTIMATE_CLONE})")
print(f"LLM: {LLM_MODEL} @ {LLM_API_BASE}")
print(f"Speakers: {NUM_SPEAKERS}, Cleanup: {CLEANUP}")

## 3. Pre-download heavy models (optional)

Run once to cache Whisper, Demucs, Pyannote, and VoxCPM on Google Drive. Future sessions reuse them.

In [ ]:
# Whisper (~3GB)
import whisperx
print("Caching Whisper...")
_m = whisperx.load_model("large-v2", "cuda", compute_type="float16"); del _m

# Demucs (~80MB)
import demucs.pretrained
print("Caching Demucs...")
_m = demucs.pretrained.get_model("htdemucs"); del _m

# Pyannote diarization
from whisperx.diarize import DiarizationPipeline
print("Caching Pyannote...")
_d = DiarizationPipeline(token=HF_TOKEN, device="cuda"); del _d

# VoxCPM (~8GB)
from voxcpm import VoxCPM
print(f"Caching VoxCPM {MODEL_ID}...")
_v = VoxCPM.from_pretrained(MODEL_ID, load_denoiser=False); del _v

import torch; torch.cuda.empty_cache()
print("\nAll models cached on Google Drive.")

## 4. Run on a single video

Tip: put a `{video_stem}.zh-cn.srt` next to the video to use subtitle-driven mode (verbatim text, no LLM).

In [ ]:
from voxcpm_dub_pipeline import dub_video
from google.colab import files
from pathlib import Path

# === Option 1: Upload a video ===
uploaded = files.upload()
video_path = list(uploaded.keys())[0]

# === Option 2: Google Drive path (uncomment) ===
# video_path = "/content/drive/MyDrive/videos/ted_talk.mp4"

stem = Path(video_path).stem
output_path = f"/content/{stem}_cn.mp4"

dub_video(
    video_path=video_path,
    output_path=output_path,
    work_dir=WORK_DIR,
    hf_token=HF_TOKEN,
    llm_api_key=LLM_API_KEY,
    llm_api_base=LLM_API_BASE,
    llm_model=LLM_MODEL,
    model_id=MODEL_ID,
    cfg_value=CFG_VALUE,
    inference_timesteps=INFERENCE_TIMESTEPS,
    ultimate_clone=ULTIMATE_CLONE,
    num_speakers=NUM_SPEAKERS,
    cleanup=CLEANUP,
)

from IPython.display import Video, display
display(Video(output_path, embed=True, width=640))

In [ ]:
# Download the dubbed video
from google.colab import files
files.download(output_path)

## 5. Batch mode (with optional recursion)

Set `RECURSIVE = True` to walk subdirectories (e.g. `module-01/`, `module-02/`), preserving the folder structure in the output.

In [ ]:
from voxcpm_dub_pipeline import dub_batch

INPUT_DIR  = "/content/drive/MyDrive/videos/input"   # <- your input folder
OUTPUT_DIR = "/content/drive/MyDrive/videos/output"  # <- dubbed output folder
RECURSIVE  = True                                    # recurse into subfolders

dub_batch(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    work_dir=WORK_DIR,
    recursive=RECURSIVE,
    hf_token=HF_TOKEN,
    llm_api_key=LLM_API_KEY,
    llm_api_base=LLM_API_BASE,
    llm_model=LLM_MODEL,
    model_id=MODEL_ID,
    cfg_value=CFG_VALUE,
    inference_timesteps=INFERENCE_TIMESTEPS,
    ultimate_clone=ULTIMATE_CLONE,
    num_speakers=NUM_SPEAKERS,
    cleanup=CLEANUP,
)
print(f"\nAll done! Check: {OUTPUT_DIR}")

---
## 6. Inspect intermediate results

The pipeline saves transcript/translations/vocals under `dub_workspace/` for debugging.

In [ ]:
import json, os

work_dirs = sorted(
    [d for d in os.listdir(WORK_DIR) if os.path.isdir(f"{WORK_DIR}/{d}")]
) if os.path.isdir(WORK_DIR) else []

if work_dirs:
    work_dir = f"{WORK_DIR}/{work_dirs[-1]}"
    print(f"Work directory: {work_dir}")
    print(f"Contents: {os.listdir(work_dir)}")

    trans_path = f"{work_dir}/translations.json"
    if os.path.exists(trans_path):
        with open(trans_path) as f:
            trans = json.load(f)
        print(f"\n--- Translations ({len(trans)}) ---")
        for t in trans[:10]:
            print(f"  #{t['id']} [{t.get('speaker','?')}] EN: {t['en'][:50]}")
            print(f"       ZH: {t['zh'][:50]}")
else:
    print(f"No work dirs under {WORK_DIR} yet.")